In [1]:
!pip install fastavro

In [22]:
import fastavro
import pandas as pd
import numpy as np

In [42]:
with open("/Users/sydney/Downloads/PM2_5_emissions_grid_per_hour.avro", "rb") as f:
    reader = fastavro.reader(f)
    df = pd.DataFrame(list(reader))
    
# Extract lists
x_coords = df["xCoords"].iloc[0]
y_coords = df["yCoords"].iloc[0]
timestamps = df["timestamps"].iloc[0]
no2_values = df["data"].iloc[0]["PM2_5"]
no2_values_raw = df["data"].iloc[0]["PM2_5"]
no2_array_flat = np.array(no2_values_raw)
# Reset to the simplest approach
# Try reshaping as (time, x, y) instead of (time, y, x)
no2_array = no2_array_flat.reshape(len(timestamps), len(x_coords), len(y_coords))

t_index = timestamps.index(43200)
no2_slice = no2_array[t_index]  # shape: (91, 124)

# Meshgrid stays the same
xx, yy = np.meshgrid(x_coords, y_coords)  # shape: (124, 91)

df_flat = pd.DataFrame({
    "x": xx.flatten(),
    "y": yy.flatten(),
    "PM25": no2_slice.T.flatten()  # .T to match (124, 91)
})

print(df_flat.head())
print(df_flat.shape)

gdf = gpd.GeoDataFrame(
    df_flat,
    geometry=gpd.points_from_xy(df_flat["x"], df_flat["y"]),
    crs="EPSG:25832"
)
gdf.to_file("PM25_43200.gpkg", driver="GPKG")
print("Done!")

             x          y          PM25
0  546003.0625  5797867.0  2.591164e-12
1  546103.0625  5797867.0  6.626443e-12
2  546203.0625  5797867.0  1.610830e-11
3  546303.0625  5797867.0  3.738376e-11
4  546403.0625  5797867.0  8.255636e-11
(11284, 3)
Done!
